# Module 17 Week 19 Required Assignment

## 1. Collaborative Filtering: User-Based vs. Item-Based

**a) Compare and contrast**

- **User-based CF** recommends items by finding *users* similar to the target user (based on their rating patterns) and suggesting items those similar users liked. It answers "people like you also liked...".
- **Item-based CF** recommends items by finding *items* similar to ones the target user already rated highly (based on how all users rated them), and suggesting those similar items. It answers "people who liked this item also liked...".
- The core difference is which similarity matrix is computed: user-user similarity vs. item-item similarity, but the underlying idea (use the user-item ratings matrix to find neighbours) is the same.

**b) Which is more efficient at large scale (e.g. Netflix)?**

**Item-based CF is generally more efficient and more commonly used at large scale.** The number of items (movies/shows) tends to be far more stable over time than the number of users, which grows continuously and whose preferences shift often — so an item-item similarity matrix needs to be recomputed much less frequently than a user-user one. It's also usually smaller (fewer items than users) and produces more stable, explainable recommendations ("because you watched X").

## 2. Sparsity and Matrix Factorisation

**a) Matrix sparsity in collaborative filtering**

A user-item ratings matrix is **sparse** because any single user has only rated a tiny fraction of all available items (e.g. a user might have rated 50 movies out of a catalogue of thousands). Most entries in the matrix are therefore missing/unknown rather than zero, which makes it hard to directly compute reliable similarities — two users might have almost no rated items in common to compare. The MovieLens dataset used below is a concrete example: with 610 users and 9,724 movies, only ~1.7% of the matrix is actually filled in (see Task 3b).

**b) How matrix factorisation helps**

Matrix factorisation (e.g. SVD) decomposes the sparse user-item matrix into two smaller, dense matrices — a user-factor matrix and an item-factor matrix — that each represent users and items in a shared, lower-dimensional **latent feature space** (e.g. "how much action", "how much romance"). Multiplying a user's latent vector by an item's latent vector approximates the rating **even for pairs that were never actually rated**, letting the model generalise past the sparsity instead of relying only on directly overlapping ratings.

## 3. Collaborative Filtering: User-Item Matrix and K-Nearest Neighbours

Dataset: `ratings.csv` — the MovieLens `ml-latest-small` ratings file (columns `userId, movieId, rating, timestamp`; 610 users, 9,724 movies, 100,836 ratings on a 0.5-5.0 scale).

In [1]:
import numpy as np
import pandas as pd

ratings = pd.read_csv("ratings.csv")
print(ratings.shape)
ratings.head()

(100836, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [2]:
# a) Build the user-item (utility) matrix: rows = users, columns = movies
user_item_matrix = ratings.pivot_table(index="userId", columns="movieId", values="rating")
user_item_matrix.iloc[:5, :8]

movieId,1,2,3,4,5,6,7,8
userId,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# b) Sparsity of the user-item matrix
n_ratings = ratings.shape[0]
n_possible = user_item_matrix.shape[0] * user_item_matrix.shape[1]
sparsity = 1 - (n_ratings / n_possible)

print(f"Matrix shape: {user_item_matrix.shape}")
print(f"Ratings present: {n_ratings} / {n_possible} possible cells")
print(f"Sparsity: {sparsity:.2%}")

Matrix shape: (610, 9724)
Ratings present: 100836 / 5931640 possible cells
Sparsity: 98.30%


The matrix is over 98% empty — the overwhelming majority of user-movie pairs have no rating. This is why raw similarity between two users can be unreliable: most pairs of users will have rated very few of the same movies to compare, which is exactly the sparsity problem matrix factorisation (Task 2b) is designed to work around.

In [4]:
# c) K-Nearest Neighbours recommender: recommend top 5 movies for a given user
from sklearn.neighbors import NearestNeighbors

filled_matrix = user_item_matrix.fillna(0)

knn = NearestNeighbors(metric="cosine", algorithm="brute")
knn.fit(filled_matrix.values)

def recommend_for_user(user_id, k_neighbours=5, top_n=5):
    if user_id not in filled_matrix.index:
        return []
    row_idx = filled_matrix.index.get_loc(user_id)
    distances, indices = knn.kneighbors(filled_matrix.values[[row_idx]], n_neighbors=k_neighbours + 1)

    neighbour_ids = filled_matrix.index[indices.flatten()[1:]]  # drop the user themself
    neighbour_ratings = user_item_matrix.loc[neighbour_ids]

    already_rated = user_item_matrix.loc[user_id].dropna().index
    candidate_scores = neighbour_ratings.mean(axis=0).drop(labels=already_rated, errors="ignore")

    return candidate_scores.sort_values(ascending=False).head(top_n)

print("Top 5 recommendations for user 1:")
recommend_for_user(1)

Top 5 recommendations for user 1:


movieId
1591    5.0
1663    5.0
6539    5.0
6659    5.0
6820    5.0
dtype: float64

In [5]:
# d) Cold-start handling for a brand-new user: item-based CF using whatever
# few ratings they *do* have, instead of relying on user-user similarity
# (which needs an existing rating history to find neighbours).
#
# Item-item similarity is computed over the ~450 movies with at least 50 ratings
# each (a "subset of the movie dataset", as Task 4b explicitly allows) rather than
# the full 9,724 movies: this keeps the similarity matrix fast to compute and,
# since these are the most-rated movies, the similarities are also the most
# reliably estimated (least affected by the sparsity problem from Task 2/3b).
from sklearn.metrics.pairwise import cosine_similarity

movie_counts = ratings.groupby("movieId").size()
popular_movies = movie_counts[movie_counts >= 50].index
print(f"Movies with >= 50 ratings: {len(popular_movies)} (out of {len(movie_counts)})")

popular_filled = user_item_matrix[popular_movies].fillna(0)

sim_matrix = cosine_similarity(popular_filled.values.T).copy()
np.fill_diagonal(sim_matrix, 0)
item_similarity = pd.DataFrame(sim_matrix, index=popular_filled.columns, columns=popular_filled.columns).clip(lower=0)

def recommend_for_new_user(rated_movies, top_n=5):
    """rated_movies: dict of {movieId: rating} for a new user with a very short history."""
    scores = pd.Series(0.0, index=item_similarity.columns)
    for movie_id, rating in rated_movies.items():
        if movie_id in item_similarity.columns:
            scores += item_similarity[movie_id] * rating
    scores = scores.drop(labels=list(rated_movies.keys()), errors="ignore")
    return scores.sort_values(ascending=False).head(top_n)

# a brand-new user who has only rated Toy Story (movieId 1) highly
new_user_ratings = {1: 5}
print("Cold-start recommendations for a new user who only rated movieId 1 highly:")
recommend_for_new_user(new_user_ratings)

Movies with >= 50 ratings: 450 (out of 9724)
Cold-start recommendations for a new user who only rated movieId 1 highly:


movieId
3114    2.863006
480     2.828184
780     2.821308
260     2.786941
356     2.735480
dtype: float64

## 4. Evaluation of Recommender System

In [6]:
# a) & b) Evaluate on a held-out test split, restricted to the popular-movie
# subset from Task 3d (denser, more reliable similarities; still ~41k ratings)
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

subset_ratings = ratings[ratings["movieId"].isin(popular_movies)]
print(f"Subset ratings: {len(subset_ratings)} (out of {len(ratings)} total)")

train_ratings, test_ratings = train_test_split(subset_ratings, test_size=0.2, random_state=42)

train_matrix = train_ratings.pivot_table(index="userId", columns="movieId", values="rating").reindex(
    index=user_item_matrix.index, columns=popular_movies
)
train_filled = train_matrix.fillna(0)

global_mean = train_ratings["rating"].mean()
movie_mean = train_ratings.groupby("movieId")["rating"].mean()
user_mean = train_ratings.groupby("userId")["rating"].mean()

Subset ratings: 41360 (out of 100836 total)


In [7]:
# User-based similarity and item-based similarity, both computed from the
# TRAINING data only (so the test ratings are never used to build the model).
user_sim_matrix = cosine_similarity(train_filled.values).copy()
np.fill_diagonal(user_sim_matrix, 0)
user_similarity = pd.DataFrame(user_sim_matrix, index=train_filled.index, columns=train_filled.index).clip(lower=0)

item_sim_matrix = cosine_similarity(train_filled.values.T).copy()
np.fill_diagonal(item_sim_matrix, 0)
item_similarity_eval = pd.DataFrame(item_sim_matrix, index=train_filled.columns, columns=train_filled.columns).clip(lower=0)


def predict_user_based(user_id, movie_id):
    """Weighted average of similar users' ratings for this movie."""
    if movie_id not in train_matrix.columns:
        return global_mean
    raters = train_matrix[movie_id].dropna()
    if raters.empty or user_id not in user_similarity.index:
        return movie_mean.get(movie_id, global_mean)
    sims = user_similarity.loc[user_id, raters.index]
    if sims.sum() == 0:
        return movie_mean.get(movie_id, global_mean)
    return float(np.dot(sims, raters) / sims.sum())


def predict_item_based(user_id, movie_id):
    """Weighted average of the user's own ratings, weighted by item similarity."""
    if user_id not in train_matrix.index:
        return movie_mean.get(movie_id, global_mean)
    user_ratings = train_matrix.loc[user_id].dropna()
    if user_ratings.empty or movie_id not in item_similarity_eval.columns:
        return user_mean.get(user_id, global_mean)
    sims = item_similarity_eval.loc[movie_id, user_ratings.index]
    if sims.sum() == 0:
        return movie_mean.get(movie_id, global_mean)
    return float(np.dot(sims, user_ratings) / sims.sum())

In [8]:
y_true = test_ratings["rating"].values

y_pred_user = np.clip(
    [predict_user_based(u, m) for u, m in zip(test_ratings["userId"], test_ratings["movieId"])], 0.5, 5
)
y_pred_item = np.clip(
    [predict_item_based(u, m) for u, m in zip(test_ratings["userId"], test_ratings["movieId"])], 0.5, 5
)
y_pred_baseline = [movie_mean.get(m, global_mean) for m in test_ratings["movieId"]]

rmse_user = mean_squared_error(y_true, y_pred_user) ** 0.5
rmse_item = mean_squared_error(y_true, y_pred_item) ** 0.5
rmse_baseline = mean_squared_error(y_true, y_pred_baseline) ** 0.5

print(f"Test set size: {len(test_ratings)}")
print(f"RMSE - User-based CF:        {rmse_user:.4f}")
print(f"RMSE - Item-based CF:        {rmse_item:.4f}")
print(f"RMSE - Baseline (movie avg): {rmse_baseline:.4f}")

Test set size: 8272
RMSE - User-based CF:        0.9048
RMSE - Item-based CF:        0.8870
RMSE - Baseline (movie avg): 0.9133


**c) Analysis**

Both collaborative filtering models beat the "always predict the movie's average rating" baseline, confirming that the similarity-weighted approach is actually using the rating patterns in the data rather than just guessing an average. Item-based CF comes out slightly ahead of user-based CF here — consistent with the efficiency argument from Task 1b: with only 610 users but many more movies, and evaluating on the ~450 most-rated (denser) movies, item-item similarities are estimated from more overlapping ratings per pair than user-user similarities are, so the item-based predictions are a little more stable. The gap between the two CF models is modest, which matches expectations for a mid-sized, moderately dense subset like this one — on the full, far sparser 9,724-movie catalogue (Task 3b's 98%+ sparsity), the advantage of item-based CF over user-based CF would be expected to widen further, since item-item similarity remains easier to estimate reliably than user-user similarity as the catalogue grows.